# Яндекс Музыка

<a id=content></a>
## Содержание
<br>[Описание проекта](#descr_proj)
<br>[Описание данных](#descr_data)
<br>
<br>[<b>Шаг 1. Загрузка данных и получение общей информации</b>](#1)
<br>
<br>[<b>Шаг 2. Предварительная обработка данных</b>](#2)
<br>
<br>[2.1. Переименование признаков](#2.1.)
<br>[2.2. Обработка пропущенных значений](#2.2.)
<br>[2.3. Выявление и обработка дубликатов](#2.3.)
<br>
<br>[<b>Шаг3. Проверка исследовательских гипотез</b>](#3)
<br>[3.1. Сравнение поведения пользователей двух столиц](#3.1.)
<br>[3.2. Музыка в начале и в конце недели](#3.2.)
<br>[3.3. Жанровые предпочтения в Москве и Санкт-Петербурге](#3.3.)
<br>
<br>[<b>Шаг 4. Вывод</b>](#4)

<a id=descr_proj></a>
## Описание проекта

Сравнение Москвы и Санкт-Петербурга окружено мифами. Например, Москва - мегаполис, подчинённый жёсткому ритму рабочей недели; Санкт-Петербург - культурная столица, со своими вкусами. На данных сервиса `Яндекс Музыка` необходимо провести исследование пользователей двух столиц. Целью исследования является проверка трёх гипотез:
1. Активность пользователей зависит от дня недели. Причём в Москве и Санкт-Петербруге это проявляется по-разному.
2. В понедельник утром в Москве преобладают одни жанры, а в Санкт-Петербурге - другие. Так же и вечером пятницы преобладают разные жанры - в зависимости от города.
3. Москва и Санкт-Петербург предпочитают разные жанры музыки. В Москве чащу слушают поп-музыку, в Санкт-Петербурге - русский рэп.

<a id=descr_data></a>
## Описание данных

В представленном датасете `yandex_music_project.csv` имеются данные по 65079 срокам в разрезе 7 признаков: 

|№ п/п| Наименование признака | Краткое описание                                       | Единицы измерения | Значение |
|---|:------------------------|:-------------------------------------------------------|:------------------|:---------|
|1. |`userID` | идентификатор пользователя | обезличенное кодированное значение | 41 748 уникальных значений|
|2. |`Track` | название трека | наименование музыкального трека | 47 245 уникальных значений|
|3. | `artist` | имя исполнителя  | наименование музыкального исполнителя | 43 605 уникальных значений|
|4. | `genre` | жанр | наименование музыкального жанра | 289 уникальных значений |
|5. | `City` | город пользователя | наименование города пользователя | `Moscow`, `Saint-Petersburg`|
|6. | `time` | время начала прослушивания | часы:минуты:секунды | 20 392 уникальных значений|
|7. | `Day` | день недели | день недели прослушивания трека| `Monday`, `Wednesday`, `Friday` |

Данные получены посредством последовательного применения методов `len()`, `unique()`, `min()`, `max()` к указанному датасету.

<a id=1></a>
## Шаг 1. Загрузка данных и получение общей информации

In [1]:
# импорт библиотек
import pandas as pd

In [2]:
# загрузка данных
df = pd.read_csv('yandex_music_project.csv')

In [3]:
# вывод на ознакомление первых пяти строк массива данных
df.head()

,userID,Track,artist,genre,City,time,Day
0,FFB692EC,Kamigata To Boots,The Mass Missile,rock,Saint-Petersburg,20:28:33,Wednesday
1,55204538,Delayed Because of Accident,Andreas Rönnberg,rock,Moscow,14:07:09,Friday
2,20EC38,Funiculì funiculà,Mario Lanza,pop,Saint-Petersburg,20:58:07,Wednesday
3,A3DD03C9,Dragons in the Sunset,Fire + Ice,folk,Saint-Petersburg,08:37:09,Monday
4,E2DC1FAE,Soul People,Space Echo,dance,Moscow,08:34:34,Monday


In [4]:
# обобщённые сведения
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 65079 entries, 0 to 65078
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0     userID  65079 non-null  object
 1   Track     63848 non-null  object
 2   artist    57876 non-null  object
 3   genre     63881 non-null  object
 4     City    65079 non-null  object
 5   time      65079 non-null  object
 6   Day       65079 non-null  object
dtypes: object(7)
memory usage: 3.5+ MB


**Выводы:** по итогу первичного ознакомления с данными мы можем сразу зафиксировать наличие проблем с именованием признаков, в том числе наличие пробелов в начале наименования признака, отсутствие нижнего регистра, сочетание строчных и прописных букв. Также необходимо отметить пропуски в данных по признакам `artist` (7203 пропущенных значения), `track` (1231 пропущенное значение), `genre` (1198 пропущенных значений). В признаке `time` хранятся данные типа `object`, что противоречит логике исследования. Требуется предварительная обработка и очистка данных.

<a id=2></a>
## Шаг 2. Предварительная обработка данных

В датасете признаки `userID`, `City`, `time`, `Day` описывают самого пользователя. Признаки `artist`, `Track`, `genre` описывают музыкальную композицию.

<a id=2.1.></a>
### 2.1. Переименование признаков

In [5]:
# перечень названий столбцов таблицы df
df.columns

Index(['  userID', 'Track', 'artist', 'genre', '  City  ', 'time', 'Day'], dtype='object')

In [6]:
# переименование столбцов
df = df.rename(columns={'  userID': 'user_id', 'Track': 'track', '  City  ': 'city', 'Day': 'day'})

In [7]:
# проверка результатов - перечень названий столбцов
df.columns

Index(['user_id', 'track', 'artist', 'genre', 'city', 'time', 'day'], dtype='object')

<a id=2.2.></a>
### 2.2. Обработка пропущенных значений

In [8]:
# подсчёт пропусков
df.isnull().sum()

user_id       0
track      1231
artist     7203
genre      1198
city          0
time          0
day           0
dtype: int64

Не все пропущенные значения влияют на исследование. Так, в `track` и `artist` пропуски не важны для исследования, будет достаточным заменить их явными обозначениями.
Пропуски в `genre` могут помешать сравнению музыкальных вкусов Москвы и Санкт-Петербурга. На практике было бы правильно установить причину пропусков и восстановить данные. В данном учебном проекте мы лишены такой возможности, поэтому необходимо:
* заполнить эти пропуски явными обозначениями;
* оценить, насколько они повредят расчётам.
Для этого заменим пропущенные значения в признаках `track`, `artist` и `genre` заглушкой `unknown`: создадим список `columns_to_replace`, переберём все его элементы циклом `for` и для каждого столбца произведём замену пропущенных значений.

In [9]:
# перебор названий столбцов в цикле и замена пропущенных значений на 'unknown'
columns_to_replace = ['track', 'artist', 'genre']
for column in columns_to_replace:
    df[column] = df[column].fillna('unknown')

In [10]:
# проверка на пропущенные значения
df.isnull().sum()

user_id    0
track      0
artist     0
genre      0
city       0
time       0
day        0
dtype: int64

<a id=2.3.></a>
### 2.3. Выявление и обработка дубликатов

In [11]:
# подсчёт явных дубликатов
df.duplicated().sum()

np.int64(3826)

In [12]:
# удаление явных дубликатов
df = df.drop_duplicates()

In [13]:
# проверка на отсутствие дубликатов
df.duplicated().sum()

np.int64(0)

In [14]:
# проверка на неявные дубликаты по признаку `genre`
df['genre'].sort_values().unique()

array(['acid', 'acoustic', 'action', 'adult', 'africa', 'afrikaans',
       'alternative', 'alternativepunk', 'ambient', 'americana',
       'animated', 'anime', 'arabesk', 'arabic', 'arena',
       'argentinetango', 'art', 'audiobook', 'author', 'avantgarde',
       'axé', 'baile', 'balkan', 'beats', 'bigroom', 'black', 'bluegrass',
       'blues', 'bollywood', 'bossa', 'brazilian', 'breakbeat', 'breaks',
       'broadway', 'cantautori', 'cantopop', 'canzone', 'caribbean',
       'caucasian', 'celtic', 'chamber', 'chanson', 'children', 'chill',
       'chinese', 'choral', 'christian', 'christmas', 'classical',
       'classicmetal', 'club', 'colombian', 'comedy', 'conjazz',
       'contemporary', 'country', 'cuban', 'dance', 'dancehall',
       'dancepop', 'dark', 'death', 'deep', 'deutschrock', 'deutschspr',
       'dirty', 'disco', 'dnb', 'documentary', 'downbeat', 'downtempo',
       'drum', 'dub', 'dubstep', 'eastern', 'easy', 'electronic',
       'electropop', 'emo', 'entehno', '

В полученном списке присутствуют неявные дубликаты названия жанра `hiphop`:
* hip,
* hop,
* hip-hop.

In [15]:
# Устранение неявных дубликатов
duplicates = ['hip', 'hop', 'hip-hop'] # дублирующие значения
name = 'hiphop' # правильное значение
df['genre'] = df['genre'].replace(duplicates, name)

In [16]:
# проверка устранения неявных дубликатов
df['genre'].sort_values().unique()

array(['acid', 'acoustic', 'action', 'adult', 'africa', 'afrikaans',
       'alternative', 'alternativepunk', 'ambient', 'americana',
       'animated', 'anime', 'arabesk', 'arabic', 'arena',
       'argentinetango', 'art', 'audiobook', 'author', 'avantgarde',
       'axé', 'baile', 'balkan', 'beats', 'bigroom', 'black', 'bluegrass',
       'blues', 'bollywood', 'bossa', 'brazilian', 'breakbeat', 'breaks',
       'broadway', 'cantautori', 'cantopop', 'canzone', 'caribbean',
       'caucasian', 'celtic', 'chamber', 'chanson', 'children', 'chill',
       'chinese', 'choral', 'christian', 'christmas', 'classical',
       'classicmetal', 'club', 'colombian', 'comedy', 'conjazz',
       'contemporary', 'country', 'cuban', 'dance', 'dancehall',
       'dancepop', 'dark', 'death', 'deep', 'deutschrock', 'deutschspr',
       'dirty', 'disco', 'dnb', 'documentary', 'downbeat', 'downtempo',
       'drum', 'dub', 'dubstep', 'eastern', 'easy', 'electronic',
       'electropop', 'emo', 'entehno', '

**Выводы:** на данном этапе были выявлены и частично устранены проблемы в данных. Заголовки данных были приведены к единому стилю, пропущенные значения были заполнены заглушками `unknown` ввиду невозможности заполнения синтетическими данными, а также были устранены явные и неявные дубликаты. Потери в данных составили 3826 строк или 5.87%.

<a id=3></a>
## Шаг 3. Проверка исследовательских гипотез

<a id=3.1.></a>
### 3.1. Сравнение поведения пользователей двух столиц

Первая гипотеза утверждает, что пользователи по-разному слушают музыку в Москве и Санкт-Петербурге. Проверим это предположение по данным о трёх днях недели — понедельнике, среде и пятнице. Для этого разделим пользователей Москвы и Санкт-Петербурга, а затем сравним, сколько треков прослушала каждая группа пользователей в понедельник, среду и пятницу.

In [17]:
# подсчёт количества прослушиваний в каждом городе
df.groupby('city')['user_id'].count().to_frame()

,user_id
city,
Moscow,42741
Saint-Petersburg,18512


В Москве прослушиваний больше, чем в Санкт-Петербурге. Из этого не следует, что московские пользователи чаще слушают музыку. Просто самих пользователей в Москве больше. Теперь сгруппируем данные по дню недели и посчитаем прослушивания в понедельник, среду и пятницу.

In [18]:
# подсчёт прослушиваний в каждый из трёх дней
df.groupby('day')['day'].count().to_frame()

,day
day,
Friday,21840
Monday,21354
Wednesday,18059


В среднем пользователи из двух городов менее активны по средам. Но картина может измениться, если рассмотреть каждый город в отдельности.

<br>Мы применили группировку по городу и по дням недели. Теперь напишем функцию, которая объединит два эти расчёта. Создадим функцию `number_tracks()`, которая посчитает прослушивания для заданного дня и города. Ей понадобятся два параметра: день недели и название города. В функции сохраним в переменную строки исходной таблицы, у которых значение в колонке `day` равно параметру `day`, в колонке `city` равно параметру `city`. Для этого применим последовательную фильтрацию с логической индексацией, затем посчитаем значения в столбце `user_id` получившейся таблицы. Результат сохраним в новую переменную. Вернём эту переменную из функции.

In [19]:
# функция для подсчёта прослушиваний для конкретного города и дня.
# с помощью последовательной фильтрации с логической индексацией она 
# сначала получит из исходной таблицы строки с нужным днём,
# затем из результата отфильтрует строки с нужным городом,
# методом count() посчитает количество значений в колонке user_id. 
# это количество функция вернёт в качестве результата

def number_tracks(day, city):
    track_list = df[(df['day'] == day) & (df['city'] == city)]
    track_list_count = track_list['user_id'].count()
    return track_list_count

Вызовем функцию `number_tracks()` шесть раз, меняя значение параметров — так, чтобы получить данные для каждого города в каждый из трёх дней.

In [20]:
# количество прослушиваний в Москве по понедельникам
number_tracks('Monday', 'Moscow')

np.int64(15740)

In [21]:
# количество прослушиваний в Санкт-Петербурге по понедельникам
number_tracks('Monday', 'Saint-Petersburg')

np.int64(5614)

In [22]:
# количество прослушиваний в Москве по средам
number_tracks('Wednesday', 'Moscow')

np.int64(11056)

In [23]:
# количество прослушиваний в Санкт-Петербурге по средам
number_tracks('Wednesday', 'Saint-Petersburg')

np.int64(7003)

In [24]:
# количество прослушиваний в Москве по пятницам
number_tracks('Friday', 'Moscow')

np.int64(15945)

In [25]:
# количество прослушиваний в Санкт-Петербурге по пятницам
number_tracks('Friday', 'Saint-Petersburg')

np.int64(5895)

Создадим c помощью конструктора `pd.DataFrame` таблицу, где
* названия колонок — ['city', 'monday', 'wednesday', 'friday'];
* данные — результаты, которые мы получили с помощью `number_tracks`.

In [26]:
# Обобщённая таблица с результатами
data = ['Moskow', 15740, 11056, 15945], ['Saint-Petersburg', 5614, 7003, 5895]
columns = ['city', 'monday', 'wednesday', 'friday']
pivotcity = pd.DataFrame(data=data, columns=columns)
pivotcity

,city,monday,wednesday,friday
0,Moskow,15740,11056,15945
1,Saint-Petersburg,5614,7003,5895


**Выводы:** данные показывают разницу поведения пользователей:
- В Москве пик прослушиваний приходится на понедельник и пятницу, а в среду заметен спад.
- В Петербурге, наоборот, больше слушают музыку по средам. Активность в понедельник и пятницу здесь почти в равной мере уступает среде.

Таким образом, мы получили подтверждение гипотезы о зависимости активности пользователей от дня недели с различными проявлениями в каждой из геолокаций.

<a id=3.2.></a>
### 3.2. Музыка в начале и в конце недели

Согласно второй гипотезе, утром в понедельник в Москве преобладают одни жанры, а в Петербурге — другие. Так же и вечером пятницы преобладают разные жанры — в зависимости от города.

In [27]:
# выборка только по Москве
moscow_general = df[df['city'] == 'Moscow']

In [28]:
# выборка только по Санкт-Петербургу
spb_general = df[df['city'] == 'Saint-Petersburg']

Создадим функцию `genre_weekday()` с четырьмя параметрами:

* таблица (датафрейм) с данными,
* день недели,
* начальная временная метка в формате 'hh:mm',
* последняя временная метка в формате 'hh:mm'.

Функция должна вернуть информацию о топ-10 жанров тех треков, которые прослушивали в указанный день, в промежутке между двумя отметками времени.

In [29]:
# функция для подсчёта топ-10 жанров в зависимости от указанного дня недели
def genre_weekday(df, day, time1, time2):
    # последовательная фильтрация
    # оставляем в genre_df только те строки df, у которых день равен day
    genre_df = df[df['day'] == day]
    # оставляем в genre_df только те строки genre_df, у которых время меньше time2
    genre_df = genre_df[genre_df['time'] < time2]
    # оставляем в genre_df только те строки genre_df, у которых время больше time1
    genre_df = genre_df[genre_df['time'] > time1]
    # сгруппируем отфильтрованный датафрейм по столбцу с названиями жанров
    # возьмём столбец genre и посчитаем кол-во строк для каждого жанра методом count()
    genre_df_grouped = genre_df.groupby('genre')['genre'].count()
    # отсортируем результат по убыванию (чтобы в начале Series оказались самые популярные жанры)
    genre_df_sorted = genre_df_grouped.sort_values(ascending = False)
    # вернём Series с 10 самыми популярными жанрами в указанный отрезок времени заданного дня
    return genre_df_sorted[:10]

Cравним результаты функции `genre_weekday()` для Москвы и Санкт-Петербурга в понедельник утром (с 7:00 до 11:00) и в пятницу вечером (с 17:00 до 23:00):

In [30]:
# расчёт для утра понедельника в Москве
# объекты, хранящие время, являются строками и сравниваются как строки
genre_weekday(moscow_general, 'Monday', '07:00', '11:00')

genre
pop            781
dance          549
electronic     480
rock           474
hiphop         286
ruspop         186
world          181
rusrap         175
alternative    164
unknown        161
Name: genre, dtype: int64

In [31]:
# расчёт для утра понедельника в Петербурге
genre_weekday(spb_general, 'Monday', '07:00', '11:00')

genre
pop            218
dance          182
rock           162
electronic     147
hiphop          80
ruspop          64
alternative     58
rusrap          55
jazz            44
classical       40
Name: genre, dtype: int64

In [32]:
# расчёт для вечера пятницы в Москве
genre_weekday(moscow_general, 'Friday', '17:00', '23:00')

genre
pop            713
rock           517
dance          495
electronic     482
hiphop         273
world          208
ruspop         170
alternative    163
classical      163
rusrap         142
Name: genre, dtype: int64

In [33]:
# расчёт для вечера пятницы в Санкт-Петербурге
genre_weekday(spb_general, 'Friday', '17:00', '23:00')

genre
pop            256
rock           216
electronic     216
dance          210
hiphop          97
alternative     63
jazz            61
classical       60
rusrap          59
world           54
Name: genre, dtype: int64

**Выводы:** расчёт данных по топ-10 в столицах свидетельствует о большой схожести музыкальных вкусов и предпочтений за единственным отличием вхождения в московский рейтинг жанра `world` и присутствием в санкт-петербургском топе `джаза` и `классики`. Слабым местом в гипотезе остаётся присутствие тени пропусков, характерного для московского утра понедельника. В этой точке `unknown` закрывает собой 161 пропуск в исходных данных, и с полными данными этот рейтинг мог бы выглядеть совершенно иначе. Пропущенные значения занимают существенную долю в данных и угрожают достоверности исследования. Вечер пятницы не меняет эту картину. Некоторые жанры поднимаются немного выше, другие спускаются, но в целом топ-10 остаётся тем же самым.

Таким образом, вторая гипотеза подтвердилась лишь частично:
* пользователи слушают похожую музыку в начале недели и в конце.
* разница между Москвой и Петербургом не слишком выражена. В Москве чаще слушают русскую популярную музыку, в Петербурге — джаз.

Однако пропуски в данных ставят под сомнение этот результат.

<a id=3.3.></a>
### 3.3. Жанровые предпочтения в Москве и Санкт-Петербурге

Третья гипотеза о поведении пользователей сервиса `Яндекс Музыка` предполагает доминирование русского рэпа в Санкт-Петербурге, и поп-музыки в Москве - городе контрастов.

Сгруппируем таблицы `moscow_general` и `spb_general` по жанру и посчитаем прослушивания треков каждого жанра методом `count()`, отсортируем полученные результаты в порядке убывания.

In [34]:
# одной строкой: группировка таблицы moscow_general по столбцу 'genre', 
# подсчёт числа значений 'genre' в этой группировке методом count(), 
# сортировка получившегося Series в порядке убывания и сохранение в moscow_genres
moscow_genres = moscow_general.groupby('genre')['genre'].count().sort_values(ascending = False)

# одной строкой: группировка таблицы spb_general по столбцу 'genre', 
# подсчёт числа значений 'genre' в этой группировке методом count(), 
# сортировка получившегося Series в порядке убывания и сохранение в spb_genres
spb_genres = spb_general.groupby('genre')['genre'].count().sort_values(ascending = False)

In [35]:
# топ-10 московских жанров
moscow_genres[:10]

genre
pop            5892
dance          4435
rock           3965
electronic     3786
hiphop         2096
classical      1616
world          1432
alternative    1379
ruspop         1372
rusrap         1161
Name: genre, dtype: int64

In [36]:
# топ-10 санкт-петербургских жанров
spb_genres[:10]

genre
pop            2431
dance          1932
rock           1879
electronic     1736
hiphop          960
alternative     649
classical       646
rusrap          564
ruspop          538
world           515
Name: genre, dtype: int64

**Выводы**: гипотеза не подтвердилась. Поп-музыка — самый популярный жанр в Москве, как и предполагала гипотеза. Более того, в топ-10 жанров встречается близкий жанр — русская популярная музыка. Вопреки ожиданиям, рэп одинаково популярен в Москве и Петербурге. 

<a id=4></a>
## Шаг 4. Вывод

Нами были выполнены расчёты с целью проверки трёх исследовательских гипотез о поведении пользователей сервиса `Яндекс Музыка`, в ходе которых было установлено следующее:

1. День недели по-разному влияет на активность пользователей в Москве и Петербурге. 
Первая гипотеза полностью подтвердилась.

2. Музыкальные предпочтения не сильно меняются в течение недели — будь то Москва или Петербург. Небольшие различия заметны в начале недели, по понедельникам:
* в Москве слушают музыку жанра `world`,
* в Петербурге — `джаз` и `классику`.
Таким образом, вторая гипотеза подтвердилась лишь отчасти. Этот результат мог оказаться иным, если бы не пропуски в данных.

3. Во вкусах пользователей Москвы и Петербурга больше общего чем различий. Вопреки ожиданиям, предпочтения жанров в Петербурге напоминают московские.
Третья гипотеза не подтвердилась. Если различия в предпочтениях и существуют, на основной массе пользователей они незаметны.

Из данных одного сервиса едва ли можно сделать вывод о всех жителях города, особенно, когда речь идёт о государственной и культурной столицах. Однако мы получили представление о музыкальных вкусах и контрастах музыкальных жанров на протяжении того или иного временного интервала в Москве и Санкт-Петербурге.